# weewx / forecast / roof

Predict the roof sensor's actual temperature from the Bureau of Meteorology forecast and the recent difference between the two. A placeholder: the sections below are the stages still to be built.

| Term | In This Notebook |
|---|---|
| **Intent** | Adjust the forecast up or down for calibration and local conditions, using the recent roof readings against the forecasts made for them |
| **Project Flavour** | `onnx` (`model.md` 3.2) |
| **Data Splits** | `training` and `validation` the roof readings with the forecasts issued for them; `backtest` whole recorded days replayed; `golden` fixed samples and a missing input |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** load `env` and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
from workbench.data import golden_data
from workbench.env import init_env
from workbench.project import scratch_project_dir

env = init_env()

WORKSPACE = "weewx/forecast/roof"
MODEL_NAME = "weewx_forecast_roof"
TARGETS = [{"kind": "served"}]
GOLDEN = "forecast_roof_golden"
CONSTANTS = {"weewx_forecast_out": 1}

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** load the `training` and `validation` data.
- **Blocked:** needs a read-only query of `influxdb3` for `roof_temperature`, the Bureau forecasts `bom_darlington_temp_max_0` to `_7` and `_min_0` to `_7`, and solar radiation, about 79 days from 2026-08-13, and the owning module of those entities confirmed.

## Explore

- **Stage:** 2, explore.
- **Intent:** plot and check the captured data; nothing reported from it ships (R4).
- **Blocked:** needs the captured data. Plot the forecast error by horizon and hour and choose the features.

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the model on `training` only (R6).
- **Blocked:** needs the captured data. `skl2onnx` is pinned, so the fit is a scikit-learn regression converted to `onnx` with the recent error carried as state.

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** check the fit on `validation` and `backtest` through the shipped runtime (R4).
- **Blocked:** needs a fitted graph, the tolerance for narrowing float64 where `onnxruntime` lacks a kernel (`plans/mlmodels.md`, section 7) and an `acceptance_checks` function.

## Package

- **Stage:** 5, package.
- **Intent:** write the golden samples, build the signature and write the project (R1, R3).
- **Blocked:** the golden file is a stand-in and so is the signature. Replace them with the forecast, recent roof reading and solar inputs and the adjusted temperature output, then call `package_project`.

In [ ]:
from workbench.model import build_signature, register_preview, signature_input, signature_output

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {"weewx_forecast_in": [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(MODEL_NAME, [signature_input("weewx_forecast_in", "none", "An input.")], [signature_output("weewx_forecast_out", "none", "An output.")])
project_dir = scratch_project_dir(WORKSPACE)
print("no model is fitted yet, so nothing is packaged")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).
- **Blocked:** nothing to register until a project is packaged. `TARGETS` is a `served` target, and the cell below fails until then.

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS)
print("registered version", version)